<!--
(C) Copyright 2026 Barcelona Supercomputing Center (BSC) - climate-data-ai-anemoi contributors.

Description: Hackathon notebook: run anemoi-inference on a checkpoint trained by the
student and evaluate the forecast against the dataset.
-->
# Hackathon: inference with your trained model

You trained a forecast model with `anemoi-training`. In this notebook you use the checkpoint to
produce forecasts and check how good they are:

```
your checkpoint (.ckpt) ──► anemoi-inference ──► forecast.nc ──► maps and scores against the dataset
```

1. **Setup**: find your checkpoint, the evaluation dataset and the GPU.
2. **Inspect the checkpoint**: what does the model need, and what does it predict?
3. **Run inference** from a start date of your choice.
4. **Load the forecast** and the verifying data.
5. **Maps**: forecast against truth.
6. **Scores**: how fast does the error grow with lead time, and does the model beat a trivial forecast?

Cells marked **TODO** are yours to fill in or change.

> **Seen or unseen data?** A forecast is only a fair test on dates the model did not train on.
> Check which dates your training used (`dataloader.training` in your training config) and compare
> with the dates you choose below.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from anemoi.datasets import open_dataset
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "model-inference.yaml").exists())
sys.path.insert(0, str(REPO / "notebooks"))
import helpers
FIG_DIR = REPO / "figures"
FIG_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path(os.environ.get("ANEMOI_OUTPUT", str(REPO / "outputs" / "inference")))


## 2. What is inside the checkpoint?

A checkpoint stores everything inference needs: the model weights, the variable roles, the normalisation
statistics and the time step. It does **not** need your training config.

## 3. Run inference

`anemoi-inference` is configured by `configs/model-inference.yaml`. Values on the command line (`key=value`) override it.

**TODO 3 — choose the start date and the lead time.** `INIT_DATE` is the *latest input time*; it must be in the evaluation dataset with the dates before it. `LEAD_TIME_HOURS` must be a multiple of the model time step and `INIT_DATE` + lead time must also be in the dataset. Start with a short forecast and lengthen it later.

Do you remember how to run command lines in a notebook?

You can play around with the `date` and `lead_time` values in the config file to produce different forecasts, saving them to different output files.

Two things to notice in the log:

- The forecast is made of `lead_time / time step` model calls: each one feeds its own prediction back as input.
- `anemoi-inference` warns that accumulated fields (`tp`) are **not accumulated from the start of the forecast**.
  That is what we want here: like in the dataset, `tp` is the precipitation of the last 6 hours at each valid time.

## 4. Load the forecast and the truth

The forecast file has one row per valid time and one column per grid point. The truth comes from the evaluation dataset at the same valid times. 

As a reference we also build the simplest possible forecast, **persistence**: "tomorrow is like today", i.e. the state at the start date repeated at every lead time. A model is only useful if it beats persistence.

**TODO 4 - Set the variables below accordingly to your forecast and truth files.**

Hint: write `INIT_DATE` as a `pd.Timestamp`, e.g. `pd.Timestamp("2015-10-25 06:00")`, not as a plain string: the map title below formats it with `{INIT_DATE:%Y-%m-%d %H:%M}`, which only works on a date object.

In [ ]:
FORECAST_FILE = OUTPUT_DIR / "forecast.nc"
EVAL_DATASET = Path(os.environ["ANEMOI_DATASET"])
INIT_DATE = pd.Timestamp("2014-06-15 00:00")
LEAD_TIME_HOURS = 24


In [ ]:
forecast = xr.open_dataset(FORECAST_FILE)
valid_times = forecast["time"].values
lead_hours = ((valid_times - np.datetime64(INIT_DATE)) / np.timedelta64(1, "h")).astype(int)
variables = [name for name in forecast.data_vars if name not in ("latitude", "longitude")]
latitudes = forecast["latitude"].values
longitudes = forecast["longitude"].values

truth = helpers.load_truth(EVAL_DATASET, variables, valid_times)
initial_state = helpers.load_truth(EVAL_DATASET, variables, [np.datetime64(INIT_DATE)])
persistence = {name: np.repeat(initial_state[name], len(valid_times), axis=0) for name in variables}

# Sanity checks: same grid points in the same order, no missing values
grid = open_dataset(str(EVAL_DATASET))
np.testing.assert_allclose(latitudes, grid.latitudes, atol=1e-3)
assert all(np.isfinite(forecast[name].values).all() for name in variables), "NaN in the forecast"
print(f"Variables:   {variables}")
print(f"Valid times: {valid_times[0]} ... {valid_times[-1]} ({len(valid_times)} steps)")

## 5. Maps: forecast against truth

The grid is a list of 40,320 points, not a latitude-longitude array: `helpers.plot_map_panels` draws it as a triangulated surface on a Robinson map, as in Module 2 (see `helpers.build_projected_triangulation`). Truth and forecast share the same colour scale; the right panel is the error.

**TODO 5 — look at other variables and lead times.** Change `VARIABLE` and `LEAD_HOURS`. Which variable is predicted best, and which worst? Does the error look random, or does it follow a structure (a front, the tropics, high orography, the poles)?

In [ ]:
# TODO 3: change the variable and the lead time (one of `variables` and one of `lead_hours`)
VARIABLE = "2t"
LEAD_HOURS = 24

step = list(lead_hours).index(LEAD_HOURS)
figure = helpers.plot_map_panels(
    truth[VARIABLE][step],
    forecast[VARIABLE].values[step],
    longitudes,
    latitudes,
    VARIABLE,
    title=f"{VARIABLE}, start {INIT_DATE:%Y-%m-%d %H:%M}, lead time {LEAD_HOURS} h",
)
figure.savefig(
    FIG_DIR / f"spatial_truth_forecast_error_{VARIABLE}_{LEAD_HOURS}h.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
figure.savefig(FIG_DIR / "spatial_truth_forecast_error_2t_24h.png", dpi=200, bbox_inches="tight")


## 6. Scores: how does the error grow with lead time?

The standard score is the **root mean square error (RMSE)**, averaged over the globe. A grid point at high
latitude represents a much smaller area than one at the equator, so the average must be
**weighted by latitude**: with weights proportional to `cos(latitude)`.

**TODO 6 — complete `weighted_rmse`.** Replace the two `???`:

1. one weight per grid point: `cos(latitude)` (latitudes are in degrees), divided by its mean so that the
   weights average to 1;
2. the RMSE: the square root of the mean of `weights * (forecast - truth) ** 2`.

A working version is in the hint below, open it only after trying.

In [ ]:
def weighted_rmse(
    forecast_field: np.ndarray,
    truth_field: np.ndarray,
    latitudes: np.ndarray,
) -> float:
    """Return the latitude-weighted RMSE between two fields on the same grid."""

    weights = np.cos(np.deg2rad(latitudes))
    weights = weights / weights.mean()

    return np.sqrt(
        np.mean(weights * (forecast_field - truth_field) ** 2)
    )

In [ ]:
n_variables = len(variables)
figure, axes = plt.subplots(1, n_variables, figsize=(3.2 * n_variables, 3), constrained_layout=True)
for ax, name in zip(np.atleast_1d(axes), variables):
    # convert to plotting units (the offset of °C cancels in the difference)
    truth_units = helpers.to_plot_units(name, truth[name])[0]
    forecast_units, unit = helpers.to_plot_units(name, forecast[name].values)
    persistence_units = helpers.to_plot_units(name, persistence[name])[0]

    model_rmse = [weighted_rmse(f, t, latitudes) for f, t in zip(forecast_units, truth_units)]
    persistence_rmse = [weighted_rmse(p, t, latitudes) for p, t in zip(persistence_units, truth_units)]
    ax.plot(lead_hours, model_rmse, marker="o", label="model")
    ax.plot(lead_hours, persistence_rmse, marker="o", linestyle="--", label="persistence")
    ax.set_title(name)
    ax.set_xlabel("lead time (h)")
    ax.set_ylabel(f"RMSE ({unit})")
    ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].legend()
figure.savefig(
    FIG_DIR / "rmse_model_vs_persistence.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

**Questions**

- Does the model beat persistence? At which lead times, and for which variables?
- Does the error keep growing with lead time? Why does a model that feeds back its own predictions get worse?
- Which variable has the largest error relative to persistence? Think about what the model cannot know from two time steps of a 6-hourly grid.

## 7. Go further

Some ideas, all doable by editing the cells above:

- **Several start dates.** Put sections 3 to 6 in a loop over start dates (for example every 5 days) and average the RMSE curves over them: one forecast can be lucky or unlucky. Use `helpers.load_truth` and a different `output.netcdf.path` for each date.
- **Longer forecasts.** Increase `LEAD_TIME_HOURS`. When does the model stop beating persistence?
- **More variables.** Add variables to `output.netcdf.variables` in the inference config (or on the command line: `"output.netcdf.variables=[2t,q_850,u_850]"`).
- **Another checkpoint.** Point `CHECKPOINT` to another run (more training steps, another model size) and compare the curves.
- **A time series at one point.** Find the grid point closest to a city with `np.argmin` on the distance to its latitude and longitude, and plot the forecast, the truth and persistence against time. In the notebook of Module 2 you have examples of time series plots.